# Overview

In [1]:
%matplotlib inline

import pandas as pd
import importlib
import re
import warnings
import calendar
import copy
import oceanval
import pickle

import glob
import geopandas as gpd
from oceanval.quiet_nc import nc
import os
import pickle
import numpy as np
import xarray as xr
from IPython.display import Markdown as md_markdown
from tqdm import tqdm
from IPython.core.interactiveshell import InteractiveShell
from oceanval.tidiers import tidy_info
from oceanval.map_land import land_below
InteractiveShell.ast_node_interactivity = "all"

try:
    lon_lim = None
    lat_lim = None
except:
    lon_lim = None
    lat_lim = None
if lon_lim is None:
    lon_lim = [-180, 180]
if lat_lim is None:
    lat_lim = [-90, 90]

try:
    concise = False
except:
    concise = False

# the depth ranges point matchups through the water column are summarised in
from oceanval.depths import DEFAULT_DEPTH_BINS, bin_depth, depth_labels, depth_ranges_text
try:
    depth_bins = the_depth_bins
except:
    depth_bins = DEFAULT_DEPTH_BINS
depth_levels = depth_labels(depth_bins)

# in one line
from oceanval.tidiers import fix_unit, df_display, md, md_basic

warnings.filterwarnings('ignore')

from oceanval.utils import restrict_r_to_conda
restrict_r_to_conda()

%load_ext rpy2.ipython

test_status = False


i_figure = 1
i_table = 1
stamp = nc.session_info["stamp"]
out = ".trackers/" + stamp
fast_plot = False
if not os.path.exists(".trackers"):
    os.makedirs(".trackers")
# save out as empty file
with open(out, 'w') as f:
    f.write("")

def bin_value(x, bin_res):
    return np.floor((x + bin_res / 2) / bin_res + 0.5) * bin_res - bin_res / 2

try:
    vv_name = variable
except:
    vv_name = "summary"
    variable = "summary"
Variable = variable.title()

In [2]:
import datetime
from oceanval import analysis_plots as ap

# what analyze() saved, beside the report
results_dir = "../../results"
info = ap.load(results_dir)


def years_text(analysis):
    if analysis["start"] == analysis["end"]:
        return str(analysis["start"])
    return f"{analysis['start']}-{analysis['end']}"

In [3]:
created = datetime.datetime.fromisoformat(info["created"])
count = len(info["variables"])
md(
    f"This report analyses the simulation in `{info['sim_dir']}`, on its own, with no observations. "
    f"It covers {count} variable{'s' if count != 1 else ''}, and was made with **oceanval** "
    f"{info['oceanval']} on {created.day} {created.strftime('%B %Y')}. "
    "Each variable has a page of its own, listed on the left."
)

This report analyses the simulation in `/data/thaumus2/scratch/hpo/COMFORT/baseline_archerfull/2013`, on its own, with no observations. It covers 1 variable, and was made with **oceanval** 0.6.1 on 3 November 2026. Each variable has a page of its own, listed on the left.

In [4]:
rows = []
for entry in info["variables"]:
    rows.append(
        {
            "Variable": entry["name"],
            "Model variable": entry["model_variable"].replace("+", " + "),
            "Unit": ap.unit_html(entry["units"]),
            "Files": f"{entry['pattern']} ({entry['files']})",
            "Output": entry["time_res_text"].capitalize(),
            "Dates": f"{entry['first']} to {entry['last']}",
        }
    )
pd.DataFrame(rows).style.hide(axis="index")

Variable,Model variable,Unit,Files,Output,Dates
jhgjg,B1_fB1N1p,mmol Pm-3/d,amm7_1m_**_**_ptrc_T.nc (12),Monthly,2013-01-16 to 2013-12-16


In [5]:
md(f"**Table {i_table}:** The variables analysed, the model variables holding them, and the output they are read from: the files, named with their dates left out, and how many there are.")
i_table += 1

**Table 1:** The variables analysed, the model variables holding them, and the output they are read from: the files, named with their dates left out, and how many there are.

In [6]:
rows = []
for entry in info["variables"]:
    clim = entry.get("climatology")
    phen = entry.get("phenology")
    long_term = entry.get("long_term")
    seasonal = entry.get("seasonal")
    row = {
        "Variable": entry["name"],
        "Climatology": (
            f"{years_text(clim)}: " + ", ".join(layer["title"].lower() for layer in clim["layers"].values())
            if clim else "None"
        ),
        "Phenology": f"{years_text(phen)}: surface" if phen else "None",
    }
    if any(x.get("long_term") for x in info["variables"]):
        row["Long-term series"] = (
            f"{years_text(long_term)}: " + ", ".join(layer["title"].lower() for layer in long_term["layers"].values())
            if long_term else "None"
        )
    if any(x.get("seasonal") for x in info["variables"]):
        row["Seasonal cycles"] = (
            f"{years_text(seasonal)}: " + ", ".join(layer["title"].lower() for layer in seasonal["layers"].values())
            if seasonal else "None"
        )
    if any(x.get("map_changes") for x in info["variables"]):
        changes = entry.get("map_changes")
        row["Map changes"] = (
            ", ".join(f"{period['start']}-{period['end']}" for period in changes["periods"])
            + ": " + ", ".join(layer["title"].lower() for layer in changes["layers"].values())
            if changes else "None"
        )
    if any(x.get("transects") for x in info["variables"]):
        transects = entry.get("transects")
        row["Transects"] = (
            f"{years_text(transects)}: {len(transects['lines'])}" if transects else "None"
        )
    rows.append(row)
pd.DataFrame(rows).style.hide(axis="index")

Variable,Climatology,Phenology
jhgjg,2013: surface,None


In [7]:
md(f"**Table {i_table}:** What was worked out for each variable, and over which years.")
i_table += 1

**Table 2:** What was worked out for each variable, and over which years.

In [8]:
notes = []
for entry in info["variables"]:
    for kind in ("climatology", "phenology", "long_term", "seasonal", "transects"):
        analysis = entry.get(kind)
        if analysis and analysis["dates"]["issues"]:
            notes.append(f"- **{entry['name']}**, {kind.replace('_', '-')} ({years_text(analysis)}): " + " ".join(analysis["dates"]["issues"]))
    changes = entry.get("map_changes")
    for period in changes["periods"] if changes else []:
        if period["dates"]["issues"]:
            notes.append(f"- **{entry['name']}**, map changes, {period['title'].lower()} period ({years_text(period)}): " + " ".join(period["dates"]["issues"]))
md("## Gaps in the output")
if notes:
    md("The output is missing times in the years some of the variables were analysed for. Averages are taken over the times there are.\n\n" + "\n".join(notes))
else:
    md("No times are missing from the output in the years each variable was analysed for.")

## Gaps in the output

No times are missing from the output in the years each variable was analysed for.

In [9]:
md("## Methods")
methods = [
    "**Climatology.** For each variable, the mean of each month of each year is taken first, so that a month of daily output counts as much as a month of monthly output. The climatology is the mean of each month of the year over the years chosen, and the annual mean is the mean of those monthly means. "
    "At the surface, the first value from the top of the water column that is not missing is taken, as in OceanVal's matchups. "
    "At the bottom, the deepest value that is not missing is taken (nctoolkit's `bottom` method with `choice=\"value\"`). "
    "The water column's vertical mean weights each layer by its thickness; its depth integral multiplies each layer by its thickness and sums them, so a concentration per cubic metre becomes an amount per square metre.",
    "**Phenology.** The day of the seasonal peak is found in each grid cell from the variable's daily climatology at the surface: its mean on each day of the year over the years chosen. The peak is the day with the highest mean, or the first of them if there are more than one, and days are counted from 1 January in the model's own calendar.",
]
if any(entry.get("long_term") for entry in info["variables"]):
    methods.append(
        "**Long-term time series.** The mean of each month of each year is taken first, as for the climatology, and a year's value is the mean of its months' means, "
        "so that a year with output missing is the mean of the months there are: it is shown as an open circle, with the months it is the mean of in its data file. "
        "The domain and region values are area-weighted spatial means of the monthly fields."
    )
if any(entry.get("seasonal") for entry in info["variables"]):
    methods.append(
        "**Seasonal cycles.** The mean seasonal cycle: the mean of each month of the year over the years of the series, over the domain, over each region and at each station."
    )
if any(entry.get("map_changes") for entry in info["variables"]):
    methods.append(
        "**Map temporal changes.** Each period (the historical one and one or two future ones) is mapped as the mean of its years' annual means: the mean of each month of each year is taken first, as for the climatology, then the mean of those monthly means over the years of the period, in each layer chosen. "
        "The *absolute change* is the future period's map minus the historical one's, and the *percentage change* is that as a percentage of the absolute value of the historical map, so that an increase is positive, and missing where the historical value is 0. "
        "The maps of the periods share one colour scale, and the maps of the changes share another, which is symmetric about zero."
    )
if any(entry.get("transects") for entry in info["variables"]):
    methods.append(
        "**Transects.** Each transect is a straight line that runs north-south or east-west, with the options chosen for it and the years of each. The mean of each month of each year is taken first, as for the climatology, and extracted along the line with nctoolkit's `to_transect` (bilinear interpolation onto evenly spaced points, about one for each grid cell the line crosses), at the surface (the first value from the top that is not missing) or, for the *vertical climatology* option, through the water column. "
        "The *surface climatology* is the mean of each month of the year over the years chosen, and the *vertical climatology* section is the annual mean, the mean of those monthly means, at each depth. "
        "An *absolute trend* is the slope of a straight line fitted by least squares to the annual means at the surface, per year, where a year's mean is the mean of its months' means and a year with months missing is left out; it needs at least three such years. "
        "The *absolute change* is the annual mean of the last year chosen minus that of the first, and the *percentage change* is that as a percentage of the absolute value of the first year's mean, so that an increase is positive. "
        "The annual means are worked out with nctoolkit and, if a rolling mean window of more than one year was chosen for a trend, smoothed by nctoolkit's `rolling_mean` before the trend is worked out, so that a change is then the mean of the last years in the window minus the mean of the first."
    )
if info.get("subregions"):
    which = {"nwes": "the northwest European Shelf regions", "global": "the global regions"}.get(info["subregions"], f"the regions in `{info['subregions']}`")
    methods.append(
        f"**Regions.** The regional summaries use {which}, as OceanVal's validation does: each region's map is put on the model's grid by nearest neighbour, the model's field is multiplied by it, and its area-weighted spatial mean taken. "
        "A region with no cells in the model's domain is left out."
    )
if info.get("lon_lim"):
    methods.append(
        f"**Spatial subset.** The domain and region means of the time series are of the part of the model's domain in longitudes {info['lon_lim'][0]:g} to {info['lon_lim'][1]:g} and latitudes {info['lat_lim'][0]:g} to {info['lat_lim'][1]:g}. The stations are not cut by it."
    )
if info.get("stations"):
    methods.append(
        "**Stations.** The series at a station are those of the cell of the model's grid, with values, that is nearest to it, by distance over the Earth's surface. The cells used are in the tables on each variable's page."
    )
if info.get("as_missing") is not None:
    missing = info["as_missing"]
    what = f"{missing[0]:g}" if not isinstance(missing, list) else (f"{missing[0]:g}" if missing[0] == missing[1] else f"{missing[0]:g} to {missing[1]:g}")
    thickness = "" if info.get("fvcom") else ", as were those of the cell thickness"
    methods.append(f"Values of {what} in the model's output were treated as missing{thickness}.")
# raw FVCOM output's thickness is worked out from its mesh, whatever is given
if info.get("thickness") and not info.get("fvcom"):
    if info["thickness"] == "z_level":
        methods.append("The water column is on fixed depth levels (z-levels), whose thicknesses are taken from the levels themselves.")
    else:
        methods.append(f"The cell thickness of the water column is `{info['thickness']}`, at its first time if it changes over time.")
if info.get("fvcom"):
    methods.append(
        "**FVCOM output.** The simulation is raw FVCOM output, on an unstructured mesh of triangles, which is read as it is, not regridded: "
        "its surface is its top sigma layer and its bottom the deepest sigma layer with a value, and its maps are drawn on the mesh itself, "
        "a value on the nodes shaded linearly across each triangle and one on the elements filling its triangle, with the triangles that have "
        "a value missing left out. Through the water column, each layer is weighted by its thickness, its share of the sigma levels (siglev) "
        "of the depth of the water, h + zeta, at each time (on the elements, the mean of their three nodes'): the depth integral is the sum "
        "of value times thickness, and the vertical mean that divided by the summed thickness of the layers with a value. "
        "The spatial means of the time series and the phenology's daily cycle are weighted by the areas of the mesh's cells: an element's is "
        "the area of its triangle on the Earth's surface, and a node's a third of the area of each triangle it is a corner of (the control "
        "volume FVCOM gives it). A region takes the cells whose positions (the nodes', or the centres of the elements) are in it, and a "
        "station is the nearest node or element with values. A transect's monthly means are interpolated from the mesh, as its maps shade it, "
        "onto a narrow strip of a longitude-latitude grid along the line, whose rows hold the values at the line's points, and extracted from "
        "it with nctoolkit's `to_transect`; its section through the water column is of the sigma layers, each put at its depth from its share "
        "of the sigma levels of the mean depth of the water over the months worked out."
    )
if info.get("time_shift"):
    days = info["time_shift"]
    way = "back" if days < 0 else "forward"
    methods.append(f"The model's times were moved {way} {abs(days)} day{'s' if abs(days) != 1 else ''} before they were used, as they mark the end of the period each mean covers.")
md("\n\n".join(methods))

## Methods

**Climatology.** For each variable, the mean of each month of each year is taken first, so that a month of daily output counts as much as a month of monthly output. The climatology is the mean of each month of the year over the years chosen, and the annual mean is the mean of those monthly means. At the surface, the first value from the top of the water column that is not missing is taken, as in OceanVal's matchups. At the bottom, the deepest value that is not missing is taken (nctoolkit's `bottom` method with `choice="value"`). The water column's vertical mean weights each layer by its thickness; its depth integral multiplies each layer by its thickness and sums them, so a concentration per cubic metre becomes an amount per square metre.

**Phenology.** The day of the seasonal peak is found in each grid cell from the variable's daily climatology at the surface: its mean on each day of the year over the years chosen. The peak is the day with the highest mean, or the first of them if there are more than one, and days are counted from 1 January in the model's own calendar.

Values of 0 in the model's output were treated as missing, as were those of the cell thickness.

In [10]:
# delete all objects starting with ds
for name in dir():
    if name.startswith("ds"):
        del globals()[name]
for name in dir():
    if name.endswith("mask"):
        del globals()[name]
nc.cleanup()
for ff in nc.session.get_safe():
    nc.session.remove_safe(ff)
nc.cleanup()